# فاز ۲: آماده‌سازی و تحلیل داده‌های بالینی پایش طولی زخم
### Phase 2: Clinical Data Preprocessing, Mask Encoding & Longitudinal Tracking

این نوت‌بوک فرآیند پردازش داده‌های گوشی هوشمند، اصلاح نوری تطبیقی (CLAHE)، تفکیک بافت‌های ۵ گانه زخم، و پایش طولی التیام بیمار را به صورت تعاملی نمایش می‌دهد.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.config import PROJECT_ROOT, TISSUE_CLASSES, CLASS_COLORS_RGB
from src.data.preprocessor import WoundImagePreprocessor
from src.data.mask_encoder import WoundMaskEncoder
from src.data.longitudinal_tracker import LongitudinalWoundTracker
from src.data.dataset import ProcessedWoundDataset

print("All modules loaded successfully!")

## ۱. بررسی لودر داده و تفکیک در سطح بیمار (Patient-level Split)
برای جلوگیری از نشت داده (Data Leakage) بین ویزیت‌های یک بیمار، جداسازی آموزش و ارزیابی بر اساس شناسه بیمار انجام می‌شود:

In [ ]:
train_ds, val_ds = ProcessedWoundDataset.create_patient_split(val_ratio=0.25)
print(f"تعداد نمونه‌های آموزش: {len(train_ds)}")
print(f"تعداد نمونه‌های اعتبارسنجی: {len(val_ds)}")

img, mask, sample_id = train_ds[0]
print(f"شناسه نمونه: {sample_id}")
print(f"ابعاد تصویر ورودی به مدل: {img.shape}")
print(f"ابعاد ماسک کلاس‌های بافت: {mask.shape}")

## ۲. نمایش بصری تصویر پیش‌پردازش‌شده، ماسک بافت‌ها و پالت رنگی استاندارد

In [ ]:
encoder = WoundMaskEncoder()
img_np = np.transpose(img.numpy() if hasattr(img, 'numpy') else img, (1, 2, 0))
mask_np = mask.numpy() if hasattr(mask, 'numpy') else mask
mask_rgb = encoder.class_indices_to_rgb(mask_np)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), dpi=120)
ax1.imshow(img_np)
ax1.set_title(f"Preprocessed Image ({sample_id})")
ax1.axis('off')

ax2.imshow(mask_rgb)
ax2.set_title("5-Class Tissue Mask (LUTSeg Canonical)")
ax2.axis('off')
plt.show()

## ۳. تحلیل ترکیب درصدی بافت‌ها و محاسبه شاخص التیام (Healing Index)

In [ ]:
stats = encoder.compute_tissue_composition(mask_np)
print(f"کل مساحت زخم (پیکسل): {stats['wound_pixels']:,}")
print(f"شاخص التیام (Healing Index): {stats['healing_index'] * 100:.2f}%")
print("\nدرصد حضور انواع بافت:")
for t_name, t_info in stats['tissues'].items():
    print(f"- {t_name}: {t_info['percentage_of_wound']:.2f}%")

## ۴. پایش طولی و موتور هشدار پیشگیرانه (Preventive Warning Demo)
نمودار روند تغییرات زمانی بافت‌ها برای بیمار:

In [ ]:
chart_file = PROJECT_ROOT / "results/trajectories/Patient_20_trajectory.png"
if chart_file.exists():
    traj_img = cv2.cvtColor(cv2.imread(str(chart_file)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 5), dpi=150)
    plt.imshow(traj_img)
    plt.axis('off')
    plt.title("Longitudinal Healing Trajectory (Patient 20)", fontsize=12, fontweight='bold')
    plt.show()